
# MLP Lab (9) — Scikit-learn vs Keras

This notebook solves the three MLP case studies using both **Scikit-learn** and **Keras**.

### Goals
- Achieve strong performance through validation-based hyperparameter tuning.
- Compare Scikit-learn MLP models with Keras neural networks.
- Avoid test-set leakage: the test set is used only for final evaluation.
- Report the best hyperparameters and final results.

> **Note:** Exact best values can vary slightly between runs because neural networks are stochastic. Random seeds are fixed where possible for reproducibility.


In [ ]:

# =========================
# 1. Imports & Configuration
# =========================

import os
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris, load_diabetes, load_wine
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
    mean_squared_error, r2_score
)
from sklearn.inspection import permutation_importance

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)


## Case Study 1 — Iris Species Classification
Multiclass classification with a decision-boundary visualization using sepal length and sepal width.

In [ ]:

# Load Iris
iris = load_iris()
X_iris = pd.DataFrame(iris.data, columns=iris.feature_names)
y_iris = iris.target

print("Shape:", X_iris.shape)
print("Classes:", iris.target_names)
X_iris.head()


In [ ]:

# Stratified train/test split
X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(
    X_iris, y_iris,
    test_size=0.20,
    random_state=SEED,
    stratify=y_iris
)

# Scikit-learn MLP hyperparameter search
iris_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(
        activation="relu",
        solver="adam",
        max_iter=3000,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=30,
        random_state=SEED
    ))
])

iris_param_grid = {
    "mlp__hidden_layer_sizes": [(10,), (16,), (32,), (32, 16), (64, 32)],
    "mlp__alpha": [1e-5, 1e-4, 1e-3, 1e-2],
    "mlp__learning_rate_init": [0.001, 0.003, 0.01]
}

iris_grid = GridSearchCV(
    iris_pipe,
    iris_param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

iris_grid.fit(X_train_i, y_train_i)

best_sklearn_iris = iris_grid.best_estimator_
print("Best Scikit-learn parameters:")
print(iris_grid.best_params_)
print("Best CV accuracy:", iris_grid.best_score_)


In [ ]:

# Final Scikit-learn Iris evaluation
pred_i_skl = best_sklearn_iris.predict(X_test_i)

iris_skl_results = {
    "Accuracy": accuracy_score(y_test_i, pred_i_skl),
    "Precision": precision_score(y_test_i, pred_i_skl, average="weighted"),
    "Recall": recall_score(y_test_i, pred_i_skl, average="weighted"),
    "F1": f1_score(y_test_i, pred_i_skl, average="weighted")
}

pd.Series(iris_skl_results)


In [ ]:

# =========================
# Iris — Keras
# =========================

# Convert to arrays and scale using training data only
scaler_i_keras = StandardScaler()
X_train_i_scaled = scaler_i_keras.fit_transform(X_train_i)
X_test_i_scaled = scaler_i_keras.transform(X_test_i)

# Internal validation split
X_tr_i, X_val_i, y_tr_i, y_val_i = train_test_split(
    X_train_i_scaled, y_train_i,
    test_size=0.15,
    random_state=SEED,
    stratify=y_train_i
)

def build_iris_model(hidden=(32, 16), learning_rate=0.001, dropout=0.0):
    model = keras.Sequential([
        layers.Input(shape=(4,)),
        layers.Dense(hidden[0], activation="relu"),
    ])
    if dropout > 0:
        model.add(layers.Dropout(dropout))
    if len(hidden) > 1:
        model.add(layers.Dense(hidden[1], activation="relu"))
    model.add(layers.Dense(3, activation="softmax"))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

iris_keras_candidates = [
    {"hidden": (16,), "learning_rate": 0.001, "dropout": 0.0},
    {"hidden": (32,), "learning_rate": 0.001, "dropout": 0.0},
    {"hidden": (32,16), "learning_rate": 0.001, "dropout": 0.0},
    {"hidden": (64,32), "learning_rate": 0.001, "dropout": 0.0},
    {"hidden": (32,16), "learning_rate": 0.003, "dropout": 0.0},
]

best_keras_iris = None
best_keras_iris_config = None
best_val_acc = -np.inf

for config in iris_keras_candidates:
    tf.keras.backend.clear_session()
    np.random.seed(SEED)
    tf.random.set_seed(SEED)

    model = build_iris_model(**config)

    es = callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=30,
        restore_best_weights=True,
        mode="max"
    )

    history = model.fit(
        X_tr_i, y_tr_i,
        validation_data=(X_val_i, y_val_i),
        epochs=500,
        batch_size=16,
        callbacks=[es],
        verbose=0
    )

    current_best = max(history.history["val_accuracy"])

    if current_best > best_val_acc:
        best_val_acc = current_best
        best_keras_iris = model
        best_keras_iris_config = config

print("Best Keras parameters:", best_keras_iris_config)
print("Best validation accuracy:", best_val_acc)


In [ ]:

# Final Keras Iris evaluation
keras_prob_i = best_keras_iris.predict(X_test_i_scaled, verbose=0)
pred_i_keras = np.argmax(keras_prob_i, axis=1)

iris_keras_results = {
    "Accuracy": accuracy_score(y_test_i, pred_i_keras),
    "Precision": precision_score(y_test_i, pred_i_keras, average="weighted"),
    "Recall": recall_score(y_test_i, pred_i_keras, average="weighted"),
    "F1": f1_score(y_test_i, pred_i_keras, average="weighted")
}

pd.DataFrame({
    "Scikit-learn": iris_skl_results,
    "Keras": iris_keras_results
})


In [ ]:

# Iris decision boundary using Sepal Length + Sepal Width
X_2d = X_iris.iloc[:, :2].values
y_2d = y_iris

X_train_2d, X_test_2d, y_train_2d, y_test_2d = train_test_split(
    X_2d, y_2d,
    test_size=0.20,
    random_state=SEED,
    stratify=y_2d
)

boundary_model = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(
        hidden_layer_sizes=(32,16),
        activation="relu",
        solver="adam",
        alpha=1e-4,
        learning_rate_init=0.001,
        max_iter=3000,
        early_stopping=True,
        random_state=SEED
    ))
])

boundary_model.fit(X_train_2d, y_train_2d)

x_min, x_max = X_2d[:, 0].min() - 0.5, X_2d[:, 0].max() + 0.5
y_min, y_max = X_2d[:, 1].min() - 0.5, X_2d[:, 1].max() + 0.5

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 400),
    np.linspace(y_min, y_max, 400)
)

Z = boundary_model.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.figure(figsize=(9, 6))
plt.contourf(xx, yy, Z, alpha=0.25)
plt.scatter(
    X_2d[:, 0], X_2d[:, 1],
    c=y_2d, edgecolor="k", s=55
)
plt.xlabel("Sepal length (cm)")
plt.ylabel("Sepal width (cm)")
plt.title("Iris MLP Decision Boundary")
plt.show()


### Iris — Comparison

In [ ]:

iris_comparison = pd.DataFrame(
    [iris_skl_results, iris_keras_results],
    index=["Scikit-learn MLP", "Keras"]
)
iris_comparison


## Case Study 2 — Diabetes Progression Prediction
Regression using the sklearn Diabetes dataset. Linear Regression is used as the benchmark.

In [ ]:

diabetes = load_diabetes()
X_diab = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
y_diab = diabetes.target

X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_diab, y_diab,
    test_size=0.20,
    random_state=SEED
)

print("Training shape:", X_train_d.shape)
print("Test shape:", X_test_d.shape)


In [ ]:

# Scikit-learn MLPRegressor hyperparameter tuning
diab_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPRegressor(
        activation="relu",
        solver="adam",
        max_iter=5000,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=40,
        random_state=SEED
    ))
])

diab_param_grid = {
    "mlp__hidden_layer_sizes": [(20,10), (32,16), (64,32), (64,32,16)],
    "mlp__alpha": [1e-5, 1e-4, 1e-3, 1e-2],
    "mlp__learning_rate_init": [0.0005, 0.001, 0.003]
}

diab_grid = GridSearchCV(
    diab_pipe,
    diab_param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

diab_grid.fit(X_train_d, y_train_d)

best_sklearn_diab = diab_grid.best_estimator_

print("Best Scikit-learn parameters:")
print(diab_grid.best_params_)
print("Best CV R2:", diab_grid.best_score_)

pred_d_skl = best_sklearn_diab.predict(X_test_d)

diab_skl_results = {
    "MSE": mean_squared_error(y_test_d, pred_d_skl),
    "R2": r2_score(y_test_d, pred_d_skl)
}

pd.Series(diab_skl_results)


In [ ]:

# Linear Regression benchmark
linear_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("linear", LinearRegression())
])

linear_pipe.fit(X_train_d, y_train_d)
pred_d_linear = linear_pipe.predict(X_test_d)

diab_linear_results = {
    "MSE": mean_squared_error(y_test_d, pred_d_linear),
    "R2": r2_score(y_test_d, pred_d_linear)
}

print("Linear Regression benchmark:")
pd.Series(diab_linear_results)


In [ ]:

# =========================
# Diabetes — Keras
# =========================

scaler_d_keras = StandardScaler()
X_train_d_scaled = scaler_d_keras.fit_transform(X_train_d)
X_test_d_scaled = scaler_d_keras.transform(X_test_d)

X_tr_d, X_val_d, y_tr_d, y_val_d = train_test_split(
    X_train_d_scaled, y_train_d,
    test_size=0.15,
    random_state=SEED
)

def build_diabetes_model(hidden=(64,32), learning_rate=0.001):
    model = keras.Sequential([
        layers.Input(shape=(10,)),
        layers.Dense(hidden[0], activation="relu"),
        layers.Dense(hidden[1], activation="relu"),
        layers.Dense(1)
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="mse",
        metrics=[keras.metrics.RootMeanSquaredError(name="rmse")]
    )
    return model

diab_keras_candidates = [
    {"hidden": (20,10), "learning_rate": 0.001},
    {"hidden": (32,16), "learning_rate": 0.001},
    {"hidden": (64,32), "learning_rate": 0.001},
    {"hidden": (64,32), "learning_rate": 0.003},
    {"hidden": (64,32), "learning_rate": 0.0005},
]

best_keras_diab = None
best_keras_diab_config = None
best_val_mse = np.inf

for config in diab_keras_candidates:
    tf.keras.backend.clear_session()
    np.random.seed(SEED)
    tf.random.set_seed(SEED)

    model = build_diabetes_model(**config)

    es = callbacks.EarlyStopping(
        monitor="val_loss",
        patience=40,
        restore_best_weights=True
    )
    rlrop = callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=15,
        min_lr=1e-6
    )

    history = model.fit(
        X_tr_d, y_tr_d,
        validation_data=(X_val_d, y_val_d),
        epochs=1000,
        batch_size=16,
        callbacks=[es, rlrop],
        verbose=0
    )

    current_best = min(history.history["val_loss"])

    if current_best < best_val_mse:
        best_val_mse = current_best
        best_keras_diab = model
        best_keras_diab_config = config

print("Best Keras parameters:", best_keras_diab_config)
print("Best validation MSE:", best_val_mse)


In [ ]:

# Final Keras Diabetes evaluation
pred_d_keras = best_keras_diab.predict(X_test_d_scaled, verbose=0).ravel()

diab_keras_results = {
    "MSE": mean_squared_error(y_test_d, pred_d_keras),
    "R2": r2_score(y_test_d, pred_d_keras)
}

diab_comparison = pd.DataFrame(
    [diab_linear_results, diab_skl_results, diab_keras_results],
    index=["Linear Regression", "Scikit-learn MLP", "Keras"]
)

diab_comparison


In [ ]:

# Diabetes actual vs predicted
plt.figure(figsize=(8, 6))
plt.scatter(y_test_d, pred_d_skl, alpha=0.75, label="Scikit-learn MLP")
plt.scatter(y_test_d, pred_d_keras, alpha=0.75, label="Keras")
plt.plot(
    [y_test_d.min(), y_test_d.max()],
    [y_test_d.min(), y_test_d.max()],
    linestyle="--",
    label="Perfect prediction"
)
plt.xlabel("Actual target")
plt.ylabel("Predicted target")
plt.title("Diabetes: Actual vs Predicted")
plt.legend()
plt.show()


## Case Study 3 — Wine Classification
Multiclass classification with stratified splitting, ReLU, early stopping, confusion matrix, classification report, and permutation importance.

In [ ]:

wine = load_wine()
X_wine = pd.DataFrame(wine.data, columns=wine.feature_names)
y_wine = wine.target

print("Shape:", X_wine.shape)
print("Class distribution:")
print(pd.Series(y_wine).value_counts().sort_index())


In [ ]:

# Stratified split
X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(
    X_wine, y_wine,
    test_size=0.20,
    random_state=SEED,
    stratify=y_wine
)

# Scikit-learn MLP
wine_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(
        activation="relu",
        solver="adam",
        max_iter=3000,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=30,
        random_state=SEED
    ))
])

wine_param_grid = {
    "mlp__hidden_layer_sizes": [(16,), (32,), (32,16), (64,32), (64,32,16)],
    "mlp__alpha": [1e-5, 1e-4, 1e-3, 1e-2],
    "mlp__learning_rate_init": [0.001, 0.003, 0.01]
}

wine_grid = GridSearchCV(
    wine_pipe,
    wine_param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

wine_grid.fit(X_train_w, y_train_w)
best_sklearn_wine = wine_grid.best_estimator_

print("Best Scikit-learn parameters:")
print(wine_grid.best_params_)
print("Best CV accuracy:", wine_grid.best_score_)

pred_w_skl = best_sklearn_wine.predict(X_test_w)

wine_skl_results = {
    "Accuracy": accuracy_score(y_test_w, pred_w_skl),
    "Precision": precision_score(y_test_w, pred_w_skl, average="weighted"),
    "Recall": recall_score(y_test_w, pred_w_skl, average="weighted"),
    "F1": f1_score(y_test_w, pred_w_skl, average="weighted")
}

pd.Series(wine_skl_results)


In [ ]:

# Wine Scikit-learn classification report
print(classification_report(
    y_test_w,
    pred_w_skl,
    target_names=wine.target_names
))

cm_w_skl = confusion_matrix(y_test_w, pred_w_skl)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm_w_skl,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=wine.target_names,
    yticklabels=wine.target_names
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Wine — Scikit-learn Confusion Matrix")
plt.show()


In [ ]:

# =========================
# Wine — Keras
# =========================

scaler_w_keras = StandardScaler()
X_train_w_scaled = scaler_w_keras.fit_transform(X_train_w)
X_test_w_scaled = scaler_w_keras.transform(X_test_w)

X_tr_w, X_val_w, y_tr_w, y_val_w = train_test_split(
    X_train_w_scaled, y_train_w,
    test_size=0.15,
    random_state=SEED,
    stratify=y_train_w
)

def build_wine_model(hidden=(64,32), learning_rate=0.001, dropout=0.0):
    model = keras.Sequential([
        layers.Input(shape=(13,)),
        layers.Dense(hidden[0], activation="relu"),
    ])
    if dropout > 0:
        model.add(layers.Dropout(dropout))
    if len(hidden) > 1:
        model.add(layers.Dense(hidden[1], activation="relu"))
    model.add(layers.Dense(3, activation="softmax"))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

wine_keras_candidates = [
    {"hidden": (16,), "learning_rate": 0.001, "dropout": 0.0},
    {"hidden": (32,), "learning_rate": 0.001, "dropout": 0.0},
    {"hidden": (32,16), "learning_rate": 0.001, "dropout": 0.0},
    {"hidden": (64,32), "learning_rate": 0.001, "dropout": 0.0},
    {"hidden": (64,32), "learning_rate": 0.003, "dropout": 0.0},
]

best_keras_wine = None
best_keras_wine_config = None
best_val_w_acc = -np.inf

for config in wine_keras_candidates:
    tf.keras.backend.clear_session()
    np.random.seed(SEED)
    tf.random.set_seed(SEED)

    model = build_wine_model(**config)

    es = callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=40,
        restore_best_weights=True,
        mode="max"
    )

    history = model.fit(
        X_tr_w, y_tr_w,
        validation_data=(X_val_w, y_val_w),
        epochs=700,
        batch_size=16,
        callbacks=[es],
        verbose=0
    )

    current_best = max(history.history["val_accuracy"])

    if current_best > best_val_w_acc:
        best_val_w_acc = current_best
        best_keras_wine = model
        best_keras_wine_config = config

print("Best Keras parameters:", best_keras_wine_config)
print("Best validation accuracy:", best_val_w_acc)


In [ ]:

# Final Keras Wine evaluation
prob_w_keras = best_keras_wine.predict(X_test_w_scaled, verbose=0)
pred_w_keras = np.argmax(prob_w_keras, axis=1)

wine_keras_results = {
    "Accuracy": accuracy_score(y_test_w, pred_w_keras),
    "Precision": precision_score(y_test_w, pred_w_keras, average="weighted"),
    "Recall": recall_score(y_test_w, pred_w_keras, average="weighted"),
    "F1": f1_score(y_test_w, pred_w_keras, average="weighted")
}

print("Keras classification report:")
print(classification_report(
    y_test_w,
    pred_w_keras,
    target_names=wine.target_names
))

cm_w_keras = confusion_matrix(y_test_w, pred_w_keras)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm_w_keras,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=wine.target_names,
    yticklabels=wine.target_names
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Wine — Keras Confusion Matrix")
plt.show()

pd.DataFrame({
    "Scikit-learn": wine_skl_results,
    "Keras": wine_keras_results
})


In [ ]:

# =========================
# Wine — Permutation Importance
# =========================

perm = permutation_importance(
    best_sklearn_wine,
    X_test_w,
    y_test_w,
    scoring="accuracy",
    n_repeats=30,
    random_state=SEED,
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "Feature": X_wine.columns,
    "Importance Mean": perm.importances_mean,
    "Importance Std": perm.importances_std
}).sort_values("Importance Mean", ascending=False)

importance_df


In [ ]:

plt.figure(figsize=(10, 7))
plt.barh(
    importance_df["Feature"].iloc[::-1],
    importance_df["Importance Mean"].iloc[::-1],
    xerr=importance_df["Importance Std"].iloc[::-1]
)
plt.xlabel("Mean decrease in accuracy")
plt.ylabel("Feature")
plt.title("Wine — Permutation Feature Importance")
plt.tight_layout()
plt.show()


## Final Comparison — All Case Studies

In [ ]:

# Collect final results
final_results = pd.DataFrame({
    "Iris Scikit-learn": iris_skl_results,
    "Iris Keras": iris_keras_results,
    "Diabetes Linear Regression": diab_linear_results,
    "Diabetes Scikit-learn": diab_skl_results,
    "Diabetes Keras": diab_keras_results,
    "Wine Scikit-learn": wine_skl_results,
    "Wine Keras": wine_keras_results
})

final_results


In [ ]:

# Best parameters summary
best_parameters = {
    "Iris - Scikit-learn": iris_grid.best_params_,
    "Iris - Keras": best_keras_iris_config,
    "Diabetes - Scikit-learn": diab_grid.best_params_,
    "Diabetes - Keras": best_keras_diab_config,
    "Wine - Scikit-learn": wine_grid.best_params_,
    "Wine - Keras": best_keras_wine_config
}

for model_name, params in best_parameters.items():
    print(f"\n{model_name}")
    print(params)



## Conclusions

### Iris
The models are compared using Accuracy, Precision, Recall, and F1-score. The decision boundary is shown using the first two features.

### Diabetes
The task is regression, so MSE and R² are the correct metrics. Linear Regression provides a useful benchmark to determine whether the nonlinear MLP adds predictive value.

### Wine
The models are evaluated using Accuracy, Precision, Recall, F1-score, classification reports, and confusion matrices. Permutation importance shows which original features contribute most to predictive performance.

### Important
The "best" model is selected using validation/CV performance. The test set is kept separate until final evaluation, which prevents test-set leakage.
